# 03 - QLoRA Fine-Tuning (Colab)

Trains 3 LoRA configs (R=4, 8, 16) on **Qwen2.5-1.5B-Instruct** with 4-bit QLoRA. Produces `outputs/sft_r{4,8,16}/` adapter checkpoints.

**Before running:** `Runtime > Change runtime type > T4 GPU` (or better). 4-bit quantization (bitsandbytes) requires an actual CUDA GPU — this will not work on CPU.

**Time budget (per the project plan):** R=4 ~2.5h, R=8 ~3.0h, R=16 ~3.5h on a T4 (roughly half that on an A100) — total ~9 GPU-hours, which exceeds a single free Colab session. This notebook:
- **Strongly recommends mounting Google Drive** so checkpoints survive a disconnect and you can resume across multiple sessions.
- Lets you set `RANKS_TO_RUN` to train one rank at a time (e.g. `[4]` today, `[8]` tomorrow).
- Skips a rank if its adapter already exists in the output dir, and resumes from the last `save_steps` checkpoint if training was interrupted mid-run.

**Getting the repo/code into Colab:** the repo is at `https://github.com/Shhaurya17/efficient-slm-benchmark` (currently **private**). Either make it public, or clone with a token: `https://<TOKEN>@github.com/Shhaurya17/efficient-slm-benchmark.git`. Leave `REPO_URL` blank to fall back to inline defaults (matching the checked-in configs) with no repo needed — outputs then go to a Drive folder you specify.

In [ ]:
REPO_URL = ""  # e.g. "https://<TOKEN>@github.com/Shhaurya17/efficient-slm-benchmark.git"
USE_DRIVE = True  # strongly recommended for multi-hour training
DRIVE_WORKDIR = "/content/drive/MyDrive/efficient-slm-benchmark"

import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_WORKDIR, exist_ok=True)

REPO_DIR = os.path.join(DRIVE_WORKDIR, "repo") if USE_DRIVE else "/content/efficient-slm-benchmark"

if REPO_URL and not os.path.exists(os.path.join(REPO_DIR, ".git")):
    !git clone -q {REPO_URL} {REPO_DIR}

HAVE_REPO = os.path.exists(os.path.join(REPO_DIR, "configs", "train.yaml"))
OUTPUT_ROOT = os.path.join(REPO_DIR, "outputs") if HAVE_REPO else os.path.join(DRIVE_WORKDIR, "outputs")
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print("Repo available:", HAVE_REPO)
print("Output root:", OUTPUT_ROOT)

In [ ]:
%%capture
!pip install -q transformers>=4.44.0 accelerate>=0.33.0 peft>=0.12.0 bitsandbytes>=0.43.0 datasets>=2.20.0 pyyaml

In [ ]:
import torch

assert torch.cuda.is_available(), "4-bit QLoRA requires a CUDA GPU. Set Runtime > Change runtime type > GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

In [ ]:
import sys
import yaml

DEFAULT_MODEL_CONFIG = {"model_name": "Qwen/Qwen2.5-1.5B-Instruct", "torch_dtype": "float16"}
DEFAULT_DATA_CONFIG = {
    "dataset_size": 5000,
    "split_ratio": {"train": 0.8},
    "chat_template": "alpaca",
    "preprocessing": {"min_length": 10, "max_length": 2048},
}
DEFAULT_LORA_CONFIG = {
    "lora_configs": [
        {"rank": 4, "alpha": 8, "dropout": 0.05},
        {"rank": 8, "alpha": 16, "dropout": 0.05},
        {"rank": 16, "alpha": 32, "dropout": 0.05},
    ],
    "lora_target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
    "lora_bias": "none",
}
DEFAULT_TRAIN_CONFIG = {
    "output_dir": "./outputs",
    "num_train_epochs": 1,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 16,
    "gradient_accumulation_steps": 2,
    "learning_rate": 2e-4,
    "warmup_steps": 100,
    "logging_steps": 50,
    "eval_steps": 500,
    "save_steps": 500,
    "save_total_limit": 2,
    "optim": "paged_adamw_32bit",
    "max_grad_norm": 0.3,
    "seed": 42,
    "mixed_precision": "fp16",
    "gradient_checkpointing": True,
}

if HAVE_REPO:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))
    with open(os.path.join(REPO_DIR, "configs", "model.yaml")) as f:
        model_config = yaml.safe_load(f)
    with open(os.path.join(REPO_DIR, "configs", "data.yaml")) as f:
        data_config = yaml.safe_load(f)
    with open(os.path.join(REPO_DIR, "configs", "lora.yaml")) as f:
        lora_yaml = yaml.safe_load(f)
    with open(os.path.join(REPO_DIR, "configs", "train.yaml")) as f:
        train_config = yaml.safe_load(f)
else:
    model_config, data_config, lora_yaml, train_config = (
        DEFAULT_MODEL_CONFIG, DEFAULT_DATA_CONFIG, DEFAULT_LORA_CONFIG, DEFAULT_TRAIN_CONFIG,
    )

train_config["output_dir"] = OUTPUT_ROOT

try:
    from efficient_slm.data.loader import prepare_data, load_openassistant_subset, filter_valid_pairs, remove_duplicates, train_val_split
    from efficient_slm.training.trainer import (
        load_base_model_4bit, build_lora_config, tokenize_dataset, setup_qlora_trainer, train as run_train, save_checkpoint,
    )
except ImportError as e:
    raise RuntimeError(
        "Could not import efficient_slm. Set REPO_URL to clone the repo (this notebook relies on "
        "src/efficient_slm/data/loader.py and src/efficient_slm/training/trainer.py rather than duplicating "
        "that logic inline)."
    ) from e

print("Model:", model_config["model_name"])
print("LoRA ranks available:", [c["rank"] for c in lora_yaml["lora_configs"]])

## Prepare data

Regenerates the same processed train/val split as Phase 2 (deterministic via `seed=42`) so this notebook is self-contained even though `data/processed/*.jsonl` is gitignored.

In [ ]:
pairs = load_openassistant_subset(size=data_config["dataset_size"] * 2, seed=42)
pairs = filter_valid_pairs(pairs, min_length=data_config["preprocessing"]["min_length"], max_length=data_config["preprocessing"]["max_length"])
pairs = remove_duplicates(pairs)[: data_config["dataset_size"]]
train_pairs, val_pairs = train_val_split(pairs, ratio=data_config["split_ratio"]["train"], seed=42)
print(f"train: {len(train_pairs)}, val: {len(val_pairs)}")

## Train each LoRA rank

Set `RANKS_TO_RUN` to train a subset (e.g. `[4]`) if you don't have time for all three in one session.

In [ ]:
import gc
import json

RANKS_TO_RUN = [4, 8, 16]

training_metrics = {}
metrics_path = os.path.join(OUTPUT_ROOT, "training_metrics.json")
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        training_metrics = json.load(f)

for lora_cfg in lora_yaml["lora_configs"]:
    rank = lora_cfg["rank"]
    if rank not in RANKS_TO_RUN:
        continue

    adapter_dir = os.path.join(OUTPUT_ROOT, f"sft_r{rank}", "final")
    if os.path.exists(os.path.join(adapter_dir, "adapter_model.safetensors")):
        print(f"R={rank}: adapter already exists at {adapter_dir}, skipping")
        continue

    print(f"\n=== Training R={rank} ===")
    rank_output_dir = os.path.join(OUTPUT_ROOT, f"sft_r{rank}")
    rank_train_config = dict(train_config, output_dir=rank_output_dir)

    model, tokenizer = load_base_model_4bit(model_config["model_name"], torch_dtype=model_config.get("torch_dtype", "float16"))
    train_ds = tokenize_dataset(train_pairs, tokenizer, chat_template=data_config["chat_template"], max_seq_length=data_config["preprocessing"]["max_length"])
    eval_ds = tokenize_dataset(val_pairs, tokenizer, chat_template=data_config["chat_template"], max_seq_length=data_config["preprocessing"]["max_length"])

    peft_config = build_lora_config(
        rank=lora_cfg["rank"], alpha=lora_cfg["alpha"], dropout=lora_cfg["dropout"],
        target_modules=lora_yaml["lora_target_modules"], bias=lora_yaml["lora_bias"],
    )

    trainer = setup_qlora_trainer(model, train_ds, eval_ds, peft_config, rank_train_config, tokenizer)

    resume = any(name.startswith("checkpoint-") for name in os.listdir(rank_output_dir)) if os.path.exists(rank_output_dir) else False
    result = trainer.train(resume_from_checkpoint=resume) if resume else run_train(trainer)

    save_checkpoint(trainer, adapter_dir)

    trainable_params, all_params = trainer.model.get_nb_trainable_parameters()
    summary = trainer.metrics_callback.summary()
    summary.update({"trainable_params": trainable_params, "all_params": all_params, "rank": rank})
    training_metrics[f"sft_r{rank}"] = summary

    with open(metrics_path, "w") as f:
        json.dump(training_metrics, f, indent=2)

    print(f"R={rank} done: final_loss={summary['final_loss']:.4f}, trainable_params={trainable_params:,}")

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

training_metrics

## Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
for key, m in training_metrics.items():
    if not m.get("loss_history"):
        continue
    steps = [p["step"] for p in m["loss_history"]]
    losses = [p["loss"] for p in m["loss_history"]]
    ax.plot(steps, losses, label=key)
ax.set_xlabel("Step")
ax.set_ylabel("Training loss")
ax.set_title("QLoRA Training Curves")
ax.legend()
plt.tight_layout()
figures_dir = os.path.join(REPO_DIR, "figures") if HAVE_REPO else os.path.join(DRIVE_WORKDIR, "figures")
os.makedirs(figures_dir, exist_ok=True)
plt.savefig(os.path.join(figures_dir, "lora_training_curves.png"), dpi=150)
plt.show()

## Getting checkpoints back to your local repo

Adapter weights are intentionally gitignored (`*.safetensors`, `outputs/`) — they don't belong in git. With `USE_DRIVE = True`, everything under `outputs/sft_r{4,8,16}/final/` and `outputs/training_metrics.json` is already sitting in your Google Drive at `DRIVE_WORKDIR`. Options:
- Use Drive's desktop sync client to pull that folder onto your local machine, then copy it into the local repo's `outputs/`.
- Or download it directly from Colab: `!zip -r /content/outputs.zip {OUTPUT_ROOT}` then `from google.colab import files; files.download("/content/outputs.zip")`.

Phase 5 (merge + quantize) will read adapters from `outputs/sft_r{rank}/final/` — keep that layout when you move things around.